[← 09 - Thinking in Metrics](<09-thinking-in-metrics.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# Capstone - The Online Store: one real question, start to finish

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/capstone-online-store-analysis.ipynb)

The nine notebooks before this one each taught a piece of SQL on its own, with a small dataset built so every result was predictable. This one hands you a business question and a real database, with no answer key, and asks you to do what an analyst does: write the definitions down, get to know the tables, check the data before trusting it, build the analysis one named step at a time, and write up what you found, including what you couldn't find out. Every query uses something an earlier notebook taught, and the notebook says which one, so you can go back whenever a step feels unfamiliar.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** take an open business question from definition to written answer, check a dataset for partial periods, join fan-out, and missing values before analysing it, explain a trend with joins and window functions, measure whether customers come back without the bias that makes older customers look more loyal, and present the result as a portfolio piece.

## 0. Set up

Everything in this notebook runs on SQL Server and AdventureWorks2022, so both cells below are needed before anything else. In Google Colab, the second one installs SQL Server inside your session, which takes a few minutes the first time and prints each step as it goes; on your own machine it only checks that SQL Server is reachable, and the [Course Overview](<00-course-overview.ipynb>) covers installing it. If you'd rather read than run, every cell keeps its saved result.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


## 1. The brief

It's July 2014 at Adventure Works Cycles, the invented bike company behind AdventureWorks, and the head of online sales sends you two questions ahead of a planning meeting:

> 1. How has the online store done, month by month, and what's behind any big change?
> 2. When someone buys from us online for the first time, do they come back?

The deliverable is a short written answer with the numbers behind it, and an honest note of anything the data can't support. Nobody has said what "done", "big change", or "come back" mean, and that's normal; `09` showed that a question like this leaves decisions open, and that the query will make them whether you do or not.

## 2. Write the definitions first

Before any SQL, settle the decisions the brief leaves open, the habit from `09`. These are the ones this analysis uses, and each can be argued with, which is the point of writing them down:

| Decision | This analysis uses | Why |
| --- | --- | --- |
| Online order | `OnlineOrderFlag = 1` on `Sales.SalesOrderHeader` | It's how the company itself marks an order placed on the website |
| Revenue | `SubTotal`, the value of the goods, before tax and freight | Tax and freight aren't money the store earns from what it sells; `TotalDue` includes them |
| Month | The calendar month of `OrderDate` | The brief asks month by month |
| Came back | A second online order within 180 days of the first | "Ever came back" isn't fair to recent customers, which section 7 shows |
| Bike order | An order with at least one line in the Bikes category | The store sells bikes, components, clothing, and accessories, and bikes are the expensive ones |

One of these, "came back", will be revised once the data shows why the obvious version is misleading. Definitions change during an analysis, and that's fine, provided the final one is written down and the write-up uses it.

## 3. Get to know the tables

Six tables answer this brief. Before querying any of them, work out what one row of each represents, which is the grain from `04`:

<p align="center">
  <img src="graphics/capstone_tables.png" width="300" alt="Sales.Customer, one row per customer, links by CustomerID to Sales.SalesOrderHeader, one row per order; that links by SalesOrderID to Sales.SalesOrderDetail, one row per order line; each line links by ProductID to Production.Product, which links to ProductSubcategory and then ProductCategory">
</p>

In [3]:
%%tsql
SELECT
    'Sales.Customer' AS table_name,
    COUNT(*) AS row_count
FROM Sales.Customer
UNION ALL
SELECT
    'Sales.SalesOrderHeader',
    COUNT(*)
FROM Sales.SalesOrderHeader
UNION ALL
SELECT
    'Sales.SalesOrderDetail',
    COUNT(*)
FROM Sales.SalesOrderDetail
UNION ALL
SELECT
    'Production.Product',
    COUNT(*)
FROM Production.Product
UNION ALL
SELECT
    'Production.ProductSubcategory',
    COUNT(*)
FROM Production.ProductSubcategory
UNION ALL
SELECT
    'Production.ProductCategory',
    COUNT(*)
FROM Production.ProductCategory;

table_name,row_count
Sales.Customer,19820
Sales.SalesOrderHeader,31465
Sales.SalesOrderDetail,121317
Production.Product,504
Production.ProductSubcategory,37
Production.ProductCategory,4


About four order lines for every order, which matters in section 6. (`UNION ALL` stacks the results of several queries into one, and the column names come from the first.) If these tables were new to you, `INFORMATION_SCHEMA.COLUMNS`, from Appendix A, would list each one's columns and types; here, the diagram covers what's needed.

Next, who orders online? Splitting the orders by `OnlineOrderFlag`, and checking whether any online customer is a store:

In [4]:
%%tsql
SELECT
    h.OnlineOrderFlag,
    COUNT(*) AS orders,
    COUNT(DISTINCT h.CustomerID) AS customers,
    SUM(CASE WHEN c.StoreID IS NOT NULL THEN 1 ELSE 0 END) AS orders_from_stores
FROM Sales.SalesOrderHeader AS h
INNER JOIN Sales.Customer AS c
    ON c.CustomerID = h.CustomerID
GROUP BY
    h.OnlineOrderFlag;

OnlineOrderFlag,orders,customers,orders_from_stores
False,3806,635,3806
True,27659,18484,0


27,659 online orders from 18,484 customers, and not one from a store: the online store sells to individuals, and the 3,806 other orders come from 635 shops that resell the bikes. That's a useful fact for the write-up, because it means the online numbers describe people buying for themselves.

## 4. Check the data before you trust it

An analysis is only as good as the data under it, and the checks below take a few minutes. Each one catches a problem that would otherwise go into the write-up as a finding.

### When does the data start and stop?

In [5]:
%%tsql
SELECT
    YEAR(OrderDate) AS order_year,
    MIN(OrderDate) AS first_order,
    MAX(OrderDate) AS last_order,
    COUNT(*) AS online_orders
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 1
GROUP BY
    YEAR(OrderDate)
ORDER BY
    order_year;

order_year,first_order,last_order,online_orders
2011,2011-05-31 00:00:00,2011-12-31 00:00:00,1201
2012,2012-01-01 00:00:00,2012-12-31 00:00:00,2743
2013,2013-01-01 00:00:00,2013-12-31 00:00:00,12584
2014,2014-01-01 00:00:00,2014-06-30 00:00:00,11131


The first year starts on 31 May 2011 and the last one stops on 30 June 2014, so 2011 and 2014 are both partial years. Comparing 2014's orders with 2013's would compare six months against twelve, which is the "over what period" decision from `09` turning up on its own. Section 5 also looks at whether the first and last months are complete.

### Do the order lines add up to the orders?

Revenue can be read from the order (`SubTotal`) or from its lines (`LineTotal`), and section 6 needs the lines, so check that the two agree, order by order, and that no order is missing its lines:

In [6]:
%%tsql
WITH line_totals AS (
    SELECT
        SalesOrderID,
        SUM(LineTotal) AS lines_total
    FROM Sales.SalesOrderDetail
    GROUP BY
        SalesOrderID
)
SELECT
    COUNT(*) AS orders,
    COUNT(l.SalesOrderID) AS orders_with_lines,
    SUM(CASE WHEN ABS(h.SubTotal - l.lines_total) > 0.01 THEN 1 ELSE 0 END) AS orders_where_totals_disagree
FROM Sales.SalesOrderHeader AS h
LEFT JOIN line_totals AS l
    ON l.SalesOrderID = h.SalesOrderID;

orders,orders_with_lines,orders_where_totals_disagree
31465,31465,0


Every order has lines, and every order's lines add up to its `SubTotal` to the cent. The `LEFT JOIN` from `04` is what makes the first check possible: an order with no lines would survive it with a `NULL`, and `COUNT(l.SalesOrderID)` would come out smaller than `COUNT(*)`.

### Where are the NULLs?

Section 6 joins every order line to a product category through `ProductSubcategoryID`, and `02` showed that `NULL` is where rows go missing without a word:

In [7]:
%%tsql
SELECT
    (SELECT COUNT(*) FROM Production.Product WHERE ProductSubcategoryID IS NULL) AS products_without_subcategory,
    (SELECT COUNT(*)
     FROM Sales.SalesOrderDetail AS d
     INNER JOIN Production.Product AS p
         ON p.ProductID = d.ProductID
     WHERE p.ProductSubcategoryID IS NULL) AS order_lines_for_those_products;

products_without_subcategory,order_lines_for_those_products
209,0


209 products have no subcategory, but none of them has ever been sold, so an `INNER JOIN` from order lines to subcategories won't drop a single line. That's worth knowing rather than assuming: had the second number not been 0, every category total in section 6 would have been short, with no error to say so.

## 5. Question 1: the online store, month by month

One row per month, with revenue, orders, average order value, the change from the month before using `LAG()` from `06`, and a three-month moving average using the `ROWS` frame from `07`. The monthly totals are built in a CTE first, the `05` habit, so the window functions read like a sentence:

In [8]:
%%tsql
WITH monthly AS (
    SELECT
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1) AS month_start,
        COUNT(*) AS orders,
        SUM(SubTotal) AS revenue
    FROM Sales.SalesOrderHeader
    WHERE OnlineOrderFlag = 1
    GROUP BY
        DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1)
)
SELECT
    month_start,
    orders,
    CAST(revenue AS DECIMAL(12, 0)) AS revenue,
    CAST(revenue / orders AS DECIMAL(10, 0)) AS avg_order_value,
    CAST(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month_start))
         / LAG(revenue) OVER (ORDER BY month_start) AS DECIMAL(6, 1)) AS revenue_change_pct,
    CAST(AVG(revenue) OVER (
        ORDER BY month_start
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS DECIMAL(12, 0)) AS revenue_3_month_avg
FROM monthly
ORDER BY
    month_start;

month_start,orders,revenue,avg_order_value,revenue_change_pct,revenue_3_month_avg
2011-05-01,5,14477,2895,NULL,14477
2011-06-01,141,458911,3255,3069.9,236694
2011-07-01,156,506192,3245,10.3,326527
2011-08-01,150,485199,3235,-4.1,483434
2011-09-01,157,502074,3198,3.5,497821
2011-10-01,174,561681,3228,11.9,516318
2011-11-01,230,737840,3208,31.4,600532
2011-12-01,188,596747,3174,-19.1,632089
2012-01-01,193,614558,3184,3.0,649715
2012-02-01,182,592527,3256,-3.6,601277


Read the table top to bottom before drawing any conclusion, and four things stand out.

**The first month is one day.** May 2011 has 5 orders because the data starts on 31 May; it isn't a slow month, and it doesn't belong in a trend.

**The last month doesn't look like the others.** June 2014 has 939 orders but only about $49,000 of revenue, an average order of $52, and a 97.5 percent fall from May. A collapse like that, in the last month of the data, is far more often a sign of an incomplete extract than of a business falling off a cliff. Extension 1 in section 11 shows something stranger still: June 2014 contains no bike sales at all. The data alone can't settle what happened, so the honest move is the one a real analyst would make: leave June 2014 out of the trend, say so in the write-up, and ask whoever owns the data.

**Something changed in mid-2013.** From June to July 2013, orders almost tripled, from 533 to 1,564, yet revenue fell slightly, by 1.5 percent, because the average order dropped from about $1,614 to $542. More orders worth much less each means the store started selling something different, and section 6 finds out what.

**After that, revenue grew.** From $847,000 in July 2013 to $1.95 million in May 2014, which the moving average shows more smoothly than the month-to-month changes, since those jump around.

There's a quieter change too: the average order fell from about $3,100 to about $2,000 in June 2012, a year before the big one. Exercise 1 in the extensions asks you to find out why.

## 6. What changed in mid-2013?

The answer has to come from the products, which means joining orders to their lines, and lines to categories, which is where the join trap from `04` waits.

### First, the trap

Join orders to their lines and add up `SubTotal`, and the online revenue doubles:

In [9]:
%%tsql
SELECT
    (SELECT SUM(SubTotal) FROM Sales.SalesOrderHeader WHERE OnlineOrderFlag = 1) AS revenue_from_orders,
    SUM(h.SubTotal) AS revenue_after_joining_lines,
    COUNT(*) AS rows_after_join
FROM Sales.SalesOrderHeader AS h
INNER JOIN Sales.SalesOrderDetail AS d
    ON d.SalesOrderID = h.SalesOrderID
WHERE h.OnlineOrderFlag = 1;

revenue_from_orders,revenue_after_joining_lines,rows_after_join
29358677.2207,59227180.9207,60398


About $29.4 million becomes $59.2 million. The join produces one row per order *line*, and each line carries a copy of its order's `SubTotal`, so an order with three lines is counted three times. It's the repeated-credit problem from `04` on real data, and nothing errors. Once the rows are order lines, the revenue has to come from the lines too, from `LineTotal`, which section 4 checked adds up correctly.

### Revenue and orders by category, quarter by quarter

Whole quarters from the start of 2012 to the start of 2014 only, so no partial period sits in the table. `COUNT(DISTINCT ...)` counts orders rather than lines, because an order with two bikes is still one order:

In [10]:
%%tsql
WITH online_lines AS (
    SELECT
        h.SalesOrderID,
        DATEFROMPARTS(YEAR(h.OrderDate), (DATEPART(quarter, h.OrderDate) - 1) * 3 + 1, 1) AS quarter_start,
        k.Name AS category,
        d.LineTotal
    FROM Sales.SalesOrderHeader AS h
    INNER JOIN Sales.SalesOrderDetail AS d
        ON d.SalesOrderID = h.SalesOrderID
    INNER JOIN Production.Product AS p
        ON p.ProductID = d.ProductID
    INNER JOIN Production.ProductSubcategory AS s
        ON s.ProductSubcategoryID = p.ProductSubcategoryID
    INNER JOIN Production.ProductCategory AS k
        ON k.ProductCategoryID = s.ProductCategoryID
    WHERE h.OnlineOrderFlag = 1
      AND h.OrderDate >= '2012-01-01'
      AND h.OrderDate < '2014-04-01'
)
SELECT
    quarter_start,
    COUNT(DISTINCT SalesOrderID) AS orders,
    COUNT(DISTINCT CASE WHEN category = 'Bikes' THEN SalesOrderID END) AS orders_with_a_bike,
    CAST(SUM(CASE WHEN category = 'Bikes' THEN LineTotal ELSE 0 END) AS DECIMAL(12, 0)) AS bike_revenue,
    CAST(SUM(CASE WHEN category = 'Accessories' THEN LineTotal ELSE 0 END) AS DECIMAL(12, 0)) AS accessories_revenue,
    CAST(SUM(CASE WHEN category = 'Clothing' THEN LineTotal ELSE 0 END) AS DECIMAL(12, 0)) AS clothing_revenue
FROM online_lines
GROUP BY
    quarter_start
ORDER BY
    quarter_start;

quarter_start,orders,orders_with_a_bike,bike_revenue,accessories_revenue,clothing_revenue
2012-01-01,594,594,1913716,0,0
2012-04-01,673,673,1811874,0,0
2012-07-01,705,705,1306413,0,0
2012-10-01,771,771,1358596,0,0
2013-01-01,852,852,1506877,0,0
2013-04-01,1192,1157,1959378,16204,7153
2013-07-01,4866,1709,2709052,158300,75716
2013-10-01,5674,2550,4035566,178406,85475
2014-01-01,5847,2721,4308031,179885,88823


Until the second quarter of 2013, the online store sold bikes and nothing else; then accessories and clothing appear, orders multiply, and most of the new orders contain no bike. The category the brief would most likely ask about next, components, doesn't appear at all, because the online store has never sold one.

### When did each category start selling online?

In [11]:
%%tsql
SELECT
    k.Name AS category,
    MIN(h.OrderDate) AS first_sold_online,
    COUNT(DISTINCT h.SalesOrderID) AS orders
FROM Sales.SalesOrderHeader AS h
INNER JOIN Sales.SalesOrderDetail AS d
    ON d.SalesOrderID = h.SalesOrderID
INNER JOIN Production.Product AS p
    ON p.ProductID = d.ProductID
INNER JOIN Production.ProductSubcategory AS s
    ON s.ProductSubcategoryID = p.ProductSubcategoryID
INNER JOIN Production.ProductCategory AS k
    ON k.ProductCategoryID = s.ProductCategoryID
WHERE h.OnlineOrderFlag = 1
GROUP BY
    k.Name
ORDER BY
    first_sold_online;

category,first_sold_online,orders
Bikes,2011-05-31 00:00:00,15205
Clothing,2013-05-30 00:00:00,7461
Accessories,2013-05-30 00:00:00,18208


Accessories and clothing both went on sale online on 30 May 2013. (Within three days of that date, seven new bike models appeared online too, most of them cheaper than the existing range, which extension 1 lets you confirm.) One more number makes the change concrete, the share of online orders that contain a bike, before and after that date:

In [12]:
%%tsql
WITH bike_orders AS (
    SELECT DISTINCT
        d.SalesOrderID
    FROM Sales.SalesOrderDetail AS d
    INNER JOIN Production.Product AS p
        ON p.ProductID = d.ProductID
    INNER JOIN Production.ProductSubcategory AS s
        ON s.ProductSubcategoryID = p.ProductSubcategoryID
    WHERE s.ProductCategoryID = 1
)
SELECT
    CASE WHEN h.OrderDate < '2013-05-30' THEN '1. Before 30 May 2013' ELSE '2. From 30 May 2013' END AS period,
    COUNT(*) AS orders,
    COUNT(b.SalesOrderID) AS orders_with_a_bike,
    CAST(100.0 * COUNT(b.SalesOrderID) / COUNT(*) AS DECIMAL(5, 1)) AS pct_with_a_bike
FROM Sales.SalesOrderHeader AS h
LEFT JOIN bike_orders AS b
    ON b.SalesOrderID = h.SalesOrderID
WHERE h.OnlineOrderFlag = 1
GROUP BY
    CASE WHEN h.OrderDate < '2013-05-30' THEN '1. Before 30 May 2013' ELSE '2. From 30 May 2013' END
ORDER BY
    period;

period,orders,orders_with_a_bike,pct_with_a_bike
1. Before 30 May 2013,5428,5428,100.0
2. From 30 May 2013,22231,9777,44.0


Before 30 May 2013, every single online order contained a bike; from that day, 44 percent did. That's the answer to the first half of question 1: the store widened its range, the new products sell in small orders, so orders tripled while the average order fell to about a third. Bike sales didn't shrink to make room, either. The quarterly table shows orders with a bike growing after the launch as well, so the new range added customers rather than replacing the old ones.

Notice that the `100.0` in the last column is there for the reason `09` gave: `COUNT(b.SalesOrderID) / COUNT(*)` on its own is whole-number division, and it would report 0 for the second row.

## 7. Question 2: do online customers come back?

This needs one row per customer, with their first online order and its date, whether it contained a bike, and the date of their second order, if there was one. That's the top-one-per-group pattern from `06`, run twice: number each customer's orders with `ROW_NUMBER()`, then keep order 1 and join order 2 beside it.

Several queries will use this, so rather than repeat the CTEs each time, the cell saves it in a temporary table, with the constraints from `08b`: a primary key, so no customer can appear twice, and a `CHECK` that a second order can't come before a first. A `#` table belongs to this session only, and disappears when it ends, so nothing in AdventureWorks itself changes.

In [13]:
%%tsql
DROP TABLE IF EXISTS #customer_first_orders;

CREATE TABLE #customer_first_orders (
    CustomerID INT NOT NULL PRIMARY KEY,
    first_order_id INT NOT NULL,
    first_order_date DATE NOT NULL,
    first_order_had_bike BIT NOT NULL,
    second_order_date DATE NULL,
    CHECK (second_order_date >= first_order_date)
);

WITH online_orders AS (
    SELECT
        SalesOrderID,
        CustomerID,
        OrderDate,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY
                OrderDate,
                SalesOrderID
        ) AS order_number
    FROM Sales.SalesOrderHeader
    WHERE OnlineOrderFlag = 1
),
bike_orders AS (
    SELECT DISTINCT
        d.SalesOrderID
    FROM Sales.SalesOrderDetail AS d
    INNER JOIN Production.Product AS p
        ON p.ProductID = d.ProductID
    INNER JOIN Production.ProductSubcategory AS s
        ON s.ProductSubcategoryID = p.ProductSubcategoryID
    WHERE s.ProductCategoryID = 1
)
INSERT INTO #customer_first_orders (
    CustomerID,
    first_order_id,
    first_order_date,
    first_order_had_bike,
    second_order_date
)
SELECT
    f.CustomerID,
    f.SalesOrderID,
    f.OrderDate,
    CASE WHEN b.SalesOrderID IS NULL THEN 0 ELSE 1 END,
    s.OrderDate
FROM online_orders AS f
LEFT JOIN bike_orders AS b
    ON b.SalesOrderID = f.SalesOrderID
LEFT JOIN online_orders AS s
    ON s.CustomerID = f.CustomerID
   AND s.order_number = 2
WHERE f.order_number = 1;

18484 rows inserted


One row for each of the 18,484 online customers. If any customer had produced two rows, or a second order dated before the first, the constraints would have stopped the insert with an error, rather than let a bad row into everything that follows.

### The obvious answer, and why it's wrong

"Do they come back?" The obvious query is the share of customers who ever placed a second order, by the year they first ordered:

In [14]:
%%tsql
SELECT
    YEAR(first_order_date) AS first_order_year,
    COUNT(*) AS customers,
    COUNT(second_order_date) AS came_back,
    CAST(100.0 * COUNT(second_order_date) / COUNT(*) AS DECIMAL(5, 1)) AS pct_came_back,
    MIN(second_order_date) AS earliest_second_order
FROM #customer_first_orders
GROUP BY
    YEAR(first_order_date)
ORDER BY
    first_order_year;

first_order_year,customers,came_back,pct_came_back,earliest_second_order
2011,1201,997,83.0,2013-05-30
2012,2743,2567,93.6,2013-05-30
2013,8522,2891,33.9,2013-05-30
2014,6018,410,6.8,2014-01-06


Read naively, this says loyalty collapsed: 94 percent of 2012's customers came back, and only 7 percent of 2014's. But a customer who first ordered in March 2014 had at most four months to come back before the data ends, while one from 2011 had three years. The older groups look loyal partly because they've been watched for longer. Statisticians call this **censoring**: for recent customers the answer isn't "no" yet, it's "not yet, as far as we can see".

<p align="center">
  <img src="graphics/capstone_follow_up.png" width="760" alt="Timeline of how long each group of customers had to come back before the data ends in June 2014: about three years for 2011 customers, two for 2012, between six and eighteen months for 2013, at most six months for 2014; underneath, the fair version gives every customer the same 180 days">
</p>

The last column shows something else, which is worth stopping on: no customer placed a second online order before 30 May 2013, the day the new range launched, however long they'd been a customer. In a real company that's the kind of fact to take to someone who knows the business before building on it, because it might be true (perhaps there was nothing more for a bike buyer to buy), or it might be how the data was put together. AdventureWorks is invented sample data, so there's nobody to ask, and the write-up should simply say it.

### The fair version: the same 180 days for everyone

The fix is to give every customer the same window, 180 days from their first order, and to include only customers whose whole window falls inside reliable data. The data is trustworthy up to the end of May 2014, so the latest eligible first order is 30 November 2013, and starting on 1 June 2013 keeps the group to customers who had the full range to choose from:

In [15]:
%%tsql
SELECT
    CASE WHEN first_order_had_bike = 1 THEN 'Bike' ELSE 'No bike' END AS first_order,
    COUNT(*) AS customers,
    SUM(CASE WHEN second_order_date <= DATEADD(day, 180, first_order_date) THEN 1 ELSE 0 END) AS back_within_180_days,
    CAST(100.0 * SUM(CASE WHEN second_order_date <= DATEADD(day, 180, first_order_date) THEN 1 ELSE 0 END)
         / COUNT(*) AS DECIMAL(5, 1)) AS pct_back_within_180_days
FROM #customer_first_orders
WHERE first_order_date >= '2013-06-01'
  AND first_order_date < '2013-12-01'
GROUP BY
    first_order_had_bike
ORDER BY
    first_order;

first_order,customers,back_within_180_days,pct_back_within_180_days
Bike,1518,52,3.4
No bike,4456,910,20.4


Of the customers whose first online order contained a bike, 3.4 percent ordered again within 180 days; of those whose first order didn't, 20.4 percent did, roughly six times as often. That's a far more useful answer than "94 percent" or "7 percent", and it has a sensible reading: someone who has just bought a bike rarely needs another within six months, while someone who bought a bottle cage or a pair of gloves has plenty more to buy.

It's still a comparison, not a cause. The two groups might differ in other ways, and this analysis can't say whether a bike buyer *would* come back more if the store did something differently. The write-up should say what was measured and stop there.

## 8. Save the monthly figures

The write-up and any chart built on it need the monthly numbers, with the partial months marked rather than silently dropped, so someone checking the work can see what was excluded and why. A temporary table with constraints holds them:

In [16]:
%%tsql
DROP TABLE IF EXISTS #monthly_online;

CREATE TABLE #monthly_online (
    month_start DATE NOT NULL PRIMARY KEY,
    orders INT NOT NULL CHECK (orders > 0),
    revenue DECIMAL(12, 2) NOT NULL,
    is_complete BIT NOT NULL DEFAULT 1
);

INSERT INTO #monthly_online (
    month_start,
    orders,
    revenue
)
SELECT
    DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1),
    COUNT(*),
    SUM(SubTotal)
FROM Sales.SalesOrderHeader
WHERE OnlineOrderFlag = 1
GROUP BY
    DATEFROMPARTS(YEAR(OrderDate), MONTH(OrderDate), 1);

38 rows inserted


Now flag the two months section 5 found, with the safe-change habit from `08a`: look at exactly the rows the `WHERE` will touch before changing them.

In [17]:
%%tsql
SELECT
    month_start,
    orders,
    revenue
FROM #monthly_online
WHERE month_start IN ('2011-05-01', '2014-06-01');

month_start,orders,revenue
2011-05-01,5,14477.34
2014-06-01,939,49005.84


Two rows, the right two. Run the `UPDATE` inside a transaction, and check the count it reports before committing:

In [18]:
%%tsql
BEGIN TRANSACTION;

UPDATE #monthly_online
SET is_complete = 0
WHERE month_start IN ('2011-05-01', '2014-06-01');

2 rows updated


2 rows, as expected, so commit:

In [19]:
%%tsql
COMMIT;

With the partial months flagged, the headline comparison can use whole periods only: the twelve months before the launch against the twelve months after it, rather than 2013 against a half-finished 2014:

In [20]:
%%tsql
SELECT
    CASE WHEN month_start < '2013-06-01' THEN '1. Jun 2012 to May 2013' ELSE '2. Jun 2013 to May 2014' END AS twelve_months,
    COUNT(*) AS months,
    SUM(orders) AS orders,
    SUM(revenue) AS revenue,
    CAST(SUM(revenue) / SUM(orders) AS DECIMAL(10, 2)) AS avg_order_value
FROM #monthly_online
WHERE is_complete = 1
  AND month_start >= '2012-06-01'
GROUP BY
    CASE WHEN month_start < '2013-06-01' THEN '1. Jun 2012 to May 2013' ELSE '2. Jun 2013 to May 2014' END
ORDER BY
    twelve_months;

twelve_months,months,orders,revenue,avg_order_value
1. Jun 2012 to May 2013,12,3238,5792644.07,1788.96
2. Jun 2013 to May 2014,12,21265,16426480.35,772.47


Twelve months each, so the comparison is fair: online orders rose about six and a half times, from 3,238 to 21,265, and revenue almost tripled, from about $5.8 million to $16.4 million, while the average order fell from about $1,789 to $772.

## 9. Write it up

The analysis isn't finished until it's written down for the person who asked. A good write-up leads with the answers, gives the numbers that support them, states the definitions, and is plain about what the data couldn't show. Here's one version; yours should be in your own words.

> **The online store, June 2011 to May 2014**
>
> **1. How has it done?** In the twelve months from June 2013, the online store took 21,265 orders worth $16.4 million (before tax and freight), against 3,238 orders and $5.8 million in the twelve months before. The change came on 30 May 2013, when accessories and clothing went on sale online alongside several new bike models. Orders almost tripled within a month, while the average order fell from about $1,600 to about $540, because most new orders didn't include a bike: every online order contained one before that date, and 44 percent after it. Revenue then grew steadily, from $0.85 million in July 2013 to $1.95 million in May 2014, and orders containing a bike grew too.
>
> **2. Do first-time customers come back?** Among customers who first ordered online between June and November 2013, 20.4 percent of those whose first order had no bike ordered again within 180 days, against 3.4 percent of those whose first order included a bike.
>
> **Definitions.** Online orders are those flagged as online. Revenue is the goods value, excluding tax and freight. "Came back" means a second online order within 180 days of the first; I used a fixed window because comparing all-time repeat rates favours older customers, who have had longer to return.
>
> **Caveats.** June 2014 looks incomplete (revenue fell 97.5 percent and no bikes were sold), so I excluded it, along with May 2011, which holds a single day. No customer placed a second online order before 30 May 2013, which I couldn't explain from the data and would want to check with the data owner. The repeat-rate difference is a comparison, not evidence that one product causes the other.

## 10. Make it yours

This notebook is a worked example, and a copy of it isn't a portfolio piece. To make one:

- **Ask a different question of the same data.** Some that AdventureWorks can answer: which sales territories grew fastest, and is the growth in more customers or bigger orders? Which products are most often bought together? How does the reseller channel (`OnlineOrderFlag = 0`) behave, and why does its monthly revenue swing so much more than the online store's? Do customers who come back spend more on their second order than their first?
- **Follow the same shape:** brief, definitions, getting to know the tables, checks, analysis, saved results, and a write-up with caveats. The shape is what a reviewer is looking for, more than any single query.
- **Publish it.** A GitHub repository with the notebook and a README that gives the question, the definitions, the three or four findings, and the caveats, in that order, is enough. Many reviewers read only the README, so make it stand on its own.
- **Practise telling it.** "Walk me through a project" is among the most common interview questions there is, and the [study guide](<study-guides/capstone-online-store-analysis.md>) has model answers built on this analysis.

## 11. Extensions

Try these in new cells, then compare with the [solutions notebook](<solutions/capstone-online-store-analysis-solutions.ipynb>). Exercises 2 and 3 use `#customer_first_orders`, so run section 7 first if you've restarted the notebook.

1. **The earlier drop.** The average online order fell from about $3,100 to about $2,000 in June 2012, before any accessories existed. Find out why. (Hint: `Production.Product` has a `ProductModelID`, and `Production.ProductModel` has its name. Which bike models sold before and after 30 May 2012, and at what prices? And what does June 2014 contain?)
2. **How much does the window matter?** Rerun the fair repeat-rate comparison with windows of 30, 90, and 180 days, in one query. Does the gap between bike and no-bike first orders hold at every window?
3. **How long until they come back?** For the same customers, find the median number of days between first and second order, separately for bike and no-bike first orders, counting only customers who came back within 180 days. (`PERCENTILE_CONT` from `07` is a window function on SQL Server.)
4. **Stretch: what do they buy first?** For first orders without a bike, in the same June to November 2013 group, which five product subcategories appear in the most first orders?

## Before you hand in an analysis

- Write the definitions down before the SQL, and revise them in writing when the data changes your mind.
- Find the first and last dates, and look at the first and last periods for anything odd, before drawing a trend.
- Check that the detail adds up to the total, and that no rows go missing in a join, before using either.
- After any join to a table with more rows, ask what one row now is, and add up from that grain only.
- When comparing groups over time, give every group the same window, or the oldest group wins by default.
- Mark excluded data rather than dropping it silently, and say what you excluded and why.
- Lead the write-up with the answer, and end it with what the data couldn't tell you.

## Go deeper

- **[SQL for Data Analysis](https://www.oreilly.com/library/view/sql-for-data/9781492088776/)**, Cathy Tanimura, O'Reilly. A whole book on this notebook's kind of work, including a chapter on cohort analysis that goes much further than section 7.
- **[Censoring (statistics)](https://en.wikipedia.org/wiki/Censoring_(statistics))**, Wikipedia. The formal name for the "not yet" problem in section 7, and the methods statisticians use for it.
- **[AdventureWorks sample databases](https://learn.microsoft.com/en-us/sql/samples/adventureworks-install-configure)**, Microsoft Learn. Where the data comes from, with the other versions of the database.
- **[DATEFROMPARTS](https://learn.microsoft.com/en-us/sql/t-sql/functions/datefromparts-transact-sql)**, **[DATEADD](https://learn.microsoft.com/en-us/sql/t-sql/functions/dateadd-transact-sql)**, and **[PERCENTILE_CONT](https://learn.microsoft.com/en-us/sql/t-sql/functions/percentile-cont-transact-sql)**, Microsoft Learn. The date and percentile functions this notebook leans on.
- **Find a dataset of your own:** [Maven Analytics Data Playground](https://mavenanalytics.io/data-playground) and [Kaggle Datasets](https://www.kaggle.com/datasets) both have free business datasets suited to an analysis like this one.

For interview questions about this project, with model answers, see the [study guide](<study-guides/capstone-online-store-analysis.md>). For the best courses, books, and practice sites across the whole course, see [RESOURCES.md](RESOURCES.md).

## Where to go from here

That's the SQL foundation, start to finish. From here the curriculum splits by track, into Data Analytics, Analytics Engineering, and Data Engineering, and the [Start Here roadmap](https://aaboutdata.substack.com/p/start-here-from-beginner-to-job-ready) explains what each one covers. Whichever you pick, the shape of this notebook comes with you: define, check, analyse, and say plainly what you found and what you couldn't.

---

[← 09 - Thinking in Metrics](<09-thinking-in-metrics.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).